In [1]:
import pandas as pd
import numpy as np

import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt

from gnews import GNews
from datetime import datetime,timedelta,date

import os
import time

from transformers import BertTokenizer,BertForSequenceClassification,pipeline,AutoTokenizer,AutoModelForSequenceClassification
import torch

import yfinance as yf

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

C:\Users\Asus\PycharmProjects\NLP_Sentiment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class yf_price_ticker():
    def set_df(self,symbol: str,time:int,interval:str = '1d'):
        _t=yf.Ticker(symbol)
        _df=yf.download(tickers=symbol,interval="1d",period=f"{time}mo")
        _df=_df.droplevel(level=1, axis=1)
        _df.drop(['Open', 'High', 'Low'], axis=1, inplace=True)
        _df.index = pd.to_datetime(_df.index)
        _df['lreturns'] = np.log(_df['Close'] / _df['Close'].shift(1))
        _df['moving_avg_7d'] = _df['Close'].rolling(7).mean()
        _df['moving_avg_20d'] = _df['Close'].rolling(20).mean()
        _df['Volatility_20d']=_df['lreturns'].rolling(20).std()
        _df['Volatility_7d']=_df['lreturns'].rolling(7).std()

        return _t,_df
#NEW
class NewsAcquirer:
    def __init__(self,language:str = 'en' ,max_results:int = 100):
        self.gn=GNews(language=language,max_results=max_results)

    def acquire_news_keyword(self, topic: str, total_days: int = 60, news_per_day: int = 100,
                              start_date: date = None, end_date: date = None):
        print(topic)
        all_news = []

        # default end_date to today if not given
        if end_date is None:
            end_date = datetime.now().date()

        # default start_date based on total_days if not given
        if start_date is None:
            start_date = end_date - timedelta(days=total_days)

        num_days = (end_date - start_date).days

        for day_offset in range(num_days):
            target_date = start_date + timedelta(days=day_offset)
            next_date = target_date + timedelta(days=1)

            try:
                self.gn.start_date = (target_date.year, target_date.month, target_date.day)
                self.gn.end_date = (next_date.year, next_date.month, next_date.day)
                self.gn.max_results = news_per_day

                _ = pd.DataFrame(self.gn.get_news(topic))
                if _.empty:
                    continue

                _['published date'] = pd.to_datetime(_['published date'])
                print(_.head())
                time.sleep(1)
                _['date'] = _['published date'].dt.normalize()
                _['ticker'] = topic

                all_news.append(_)

            except Exception as e:
                print(f'Error on {target_date}: {e}')
                continue

        if all_news:
            return pd.concat(all_news, ignore_index=True)
        return pd.DataFrame()

    def acquire_news_top(self):
        return self.gn.get_top_news()

    def acquire_news_topic(self,topic:str):
        return self.gn.get_news(topic)

    #drops all rows that present in old_news
    def filter_new_news(self,new_news:pd.DataFrame,old_news:pd.DataFrame):
        x=0
        for index,title in enumerate(new_news.iloc[:,0]):
            if title in old_news.iloc[:,0].values:
                new_news.drop(index=index,axis=1,inplace=True)
                x=x+1
            else:
                pass
        print('Total news dropped:',x)

class NewsRecorder:

    def __init__(self, base_path:str = r"C:\Users\Asus\Documents\Skills\Python\Sentiment Analysis\test2"):
        self.base_path = base_path
        self.files = os.listdir(base_path)
        self.dfs=[]

    def _path_for(self, ticker: str) -> str:
        return os.path.join(self.base_path, f"{ticker}.csv")

    def record_news(self,new_news:pd.DataFrame,ticker:str,header:bool = True,mode:str = 'a'):
        new_news.to_csv(self._path_for(ticker=ticker),header=header,index=False,mode=mode)


    def load_all(self):
        print(self.files)
        self.files = [f for f in self.files if f not in ['aggregate.csv','ticker.csv','market.csv']]
        for ticker in self.files:
            if not ticker.endswith('.csv'):
                continue
            temp = pd.read_csv(self.base_path+f'\\{ticker}')
            temp['ticker'] = ticker
            self.dfs.append(temp)
        print(self.dfs)
        time.sleep(10)
        return pd.concat(self.dfs, ignore_index=True)

    def record_sentiment(self,ticker_df:pd.DataFrame,market_df:pd.DataFrame,agg_df:pd.DataFrame,header:bool = True,mode:str = 'w+'):
        ticker_df.to_csv(os.path.join(self.base_path, 'ticker.csv'), header=header, index=False, mode=mode)
        market_df.to_csv(os.path.join(self.base_path, 'market.csv'), header=header, index=True, mode=mode)
        agg_df.to_csv(os.path.join(self.base_path, 'aggregate.csv'), header=header, index=False, mode=mode)

class SentimentAnalyser:

    def __init__(self,base_path:str = r"C:\Users\Asus\Documents\Skills\Python\Sentiment Analysis\test2"):
        self.model_name="nickmuchi/deberta-v3-base-finetuned-finance-text-classification"

        self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)

        self.base_path=base_path
        self.dfs=[]
        self.df=[]

    def _acquire_sentiment(self,new_news:pd.DataFrame):

        texts = new_news["description"].fillna("").tolist()
        inputs = self.tokenizer(texts, return_tensors="pt", padding=True, truncation=True)


        all_probs = []
        all_logits = []

        label_map = self.model.config.id2label  # {0: 'positive', ...}

        batch_size=64

        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]

            inputs = self.tokenizer(batch_texts,return_tensors="pt",padding=True,truncation=True,max_length=128)

            with torch.no_grad():
                outputs = self.model(**inputs)

            logits = outputs.logits
            probs_tensor = torch.softmax(logits, dim=1)

            # process batch
            for j in range(len(batch_texts)):
                all_probs.append({label_map[k]: probs_tensor[j][k].item()for k in range(logits.shape[1])})

                all_logits.append({label_map[k] + '_logit': logits[j][k].item()for k in range(logits.shape[1])})

        # convert to DataFrame
        probs_df = pd.DataFrame(all_probs)
        logits_df = pd.DataFrame(all_logits)

        new_news = pd.concat([new_news.reset_index(drop=True), probs_df, logits_df], axis=1)
        new_news['label']= probs_df.idxmax(axis=1)
        print(new_news)
        new_news['sentiment_score'] = new_news['bullish'] - new_news['bearish']
        new_news['sentiment_logit'] = new_news['bullish_logit'] - new_news['bearish_logit']


        self.df = new_news
        return new_news


    def _acquire_sentiment_stats_ticker(self,compiled_df:pd.DataFrame):
        #get data from files calculates stats ticker by ticker and market general
        ticker_sentiment = compiled_df.groupby(['date', 'ticker']).agg({'sentiment_score': ['mean', 'count','std'],'sentiment_logit': ['mean', 'std'],
                                                                    'label': [lambda x: (((x == 'bullish').sum())/x.count())*100,
                                                                              lambda x: (((x == 'neutral').sum())/x.count())*100,
                                                                              lambda x: (((x == 'bearish').sum())/x.count())*100]})

        ticker_sentiment.columns = ['mean_sentiment', 'news_count','sentiment_volatility','mean_sentiment_logit','sentiment_volatility_logit','pert_bull',
                                    'pert_neutral','pert_bearish']
        ticker_sentiment = ticker_sentiment.reset_index()



        global_std = ticker_sentiment['mean_sentiment'].std()  # or compute a rolling/ticker-level std
        ticker_sentiment['sentiment_volatility_filled'] = ticker_sentiment['sentiment_volatility'].fillna(global_std).replace(0, global_std)

        ticker_sentiment['Zscore'] = (ticker_sentiment['mean_sentiment'] - ticker_sentiment['mean_sentiment'].mean()) / ticker_sentiment['sentiment_volatility_filled']

        ticker_sentiment['Zscore_logit']=(ticker_sentiment['mean_sentiment_logit']-ticker_sentiment['mean_sentiment_logit'].mean())/ticker_sentiment['sentiment_volatility_logit']
        ticker_sentiment['sentiment_change'] = ticker_sentiment.groupby('ticker')['mean_sentiment'].diff()
        ticker_sentiment['sentiment_change_logit'] = ticker_sentiment.groupby('ticker')['mean_sentiment_logit'].diff()

        return ticker_sentiment

    def _acquire_sentiment_stats_market(self,compiled_df:pd.DataFrame):
        print(compiled_df)

        market_sentiment = compiled_df.groupby(['date']).agg({'sentiment_score': ['mean', 'count', 'std'],'sentiment_logit': ['mean', 'std'],
                                                            'label': [lambda x: (((x == 'bullish').sum())/x.count())*100,
                                                                      lambda x: (((x == 'neutral').sum())/x.count())*100,
                                                                      lambda x: (((x == 'bearish').sum())/x.count())*100]})
        market_sentiment.columns = ['mean_sentiment', 'news_count', 'sentiment_volatility','mean_sentiment_logit', 'sentiment_volatility_logit','pert_bull',
                                    'pert_neutral','pert_bearish']

        market_sentiment.sort_values('date', ascending=True, inplace=True)
        market_sentiment['sentiment_volatility_filled'] = (market_sentiment['sentiment_volatility'].replace(0, np.nan).ffill())

        eps = 0.05

        market_sentiment['sentiment_volatility_filled'] = (market_sentiment['sentiment_volatility'].replace([np.inf,-np.inf],np.nan).mask(market_sentiment['sentiment_volatility'].abs() < eps, np.nan).ffill().bfill())

        market_sentiment['Zscore']=(market_sentiment['mean_sentiment']-market_sentiment['mean_sentiment'].mean())/market_sentiment['sentiment_volatility_filled']
        #cancel out
          # anything below this is "too small to trust" — tune based on your score scale



        market_sentiment['Zscore_logit']=(market_sentiment['mean_sentiment_logit']-market_sentiment['mean_sentiment_logit'].mean())/market_sentiment['sentiment_volatility_logit']
        market_sentiment['sentiment_change'] = market_sentiment['mean_sentiment'].diff()
        market_sentiment['sentiment_change_logit'] = market_sentiment['mean_sentiment_logit'].diff()

        return market_sentiment

    def acquire_sentiment_stats(self,loader):
        _df=loader.load_all()
        return _df,self._acquire_sentiment_stats_market(compiled_df=_df),self._acquire_sentiment_stats_ticker(compiled_df=_df)


    def acquire_sentiment(self,ticker_df):
        return self._acquire_sentiment(ticker_df)

class NewsPipeline:

    def __init__(self, base_path: str = r"C:\Users\Asus\Documents\Skills\Python\Sentiment Analysis\test2"):
        self.acquirer = NewsAcquirer()
        self.recorder = NewsRecorder(base_path)
        self.analyser = SentimentAnalyser()


    def _all_tickers(self):
            return ['SBI']

    def run_acquisition(self,s_date,e_date):
        for ticker in self._all_tickers():
            try:
                _tempdf=self.acquirer.acquire_news_keyword(start_date=s_date,end_date=e_date,topic=ticker)
                _tempdf_sentiment=self.analyser.acquire_sentiment(_tempdf)

                self.recorder.record_news(new_news=_tempdf_sentiment,ticker=ticker,header=False)
            except Exception as e:
                print(f'Error at run_acquisition {e}')
                pass
    def run_acquire_stats(self):
        return self.analyser.acquire_sentiment_stats(loader=self.recorder)

    def record_sen(self,tik,mar,agg):
        self.recorder.record_sentiment(ticker_df=tik,market_df=mar,agg_df=agg)

class Regression2:

    def __init__(self, base_path: str = r"C:\Users\Asus\Documents\Skills\Python\Sentiment Analysis\test2\Regression_Results"):
        self.pticker = {
            'Infosys': 'INFY.NS',
            'Bharat Petroleum Corporation': 'BPCL.NS',
        }
        self.yf = yf_price_ticker()
        self.base_path = base_path

    def run_regression(self, t_df,ticker):
        for tic in [ticker]:
            symbol = self.pticker[tic]


            candidates = {tic, f'{tic}.csv', symbol, f'{symbol}.csv'}
            ticker_sent = t_df[t_df['ticker'].isin(candidates)].copy()

            if ticker_sent.empty:
                print(f"[Regression] No sentiment rows matched for '{tic}' "
                      f"(tried {candidates}) — skipping.")
                continue

            ticker_sent.index = ticker_sent['date']
            ticker_sent.drop(columns=['ticker', 'date'], inplace=True)
            ticker_sent.index = pd.to_datetime(ticker_sent.index, format='mixed', dayfirst=True)

            price, price_df = self.yf.set_df(symbol, 24, 1)
            price_df.index = pd.to_datetime(price_df.index, dayfirst=True, format='mixed')

            combined = ticker_sent[['Zscore', 'sentiment_change']].merge(
                price_df[['lreturns', 'Close', 'moving_avg_7d']],
                left_index=True, right_index=True, how='outer'
            ).dropna()

            combined['Zscore_lag1'] = combined['Zscore'].shift(1)
            combined['lreturns_lag1'] = combined['lreturns'].shift(1)
            combined['moving_avg_7d_lag1'] = combined['moving_avg_7d'].shift(1)

            combined.dropna(inplace=True)
            combined = combined.replace([np.inf, -np.inf], np.nan)
            combined.dropna(subset=['Zscore'], inplace=True)

            if len(combined) < 10:
                print(f"[Regression] Not enough overlapping data for '{tic}' "
                      f"({len(combined)} rows) — skipping.")
                continue


            split_idx = int(len(combined) * 0.7)

            x_train = combined.iloc[:split_idx][['Zscore_lag1', 'lreturns_lag1', 'moving_avg_7d_lag1']]
            y_train = combined.iloc[:split_idx]['Close']
            x_test = combined.iloc[split_idx:][['Zscore_lag1', 'lreturns_lag1', 'moving_avg_7d_lag1']]
            y_test = combined.iloc[split_idx:]['Close']

            if len(x_train) == 0 or len(x_test) == 0:
                print(f"[Regression] Empty train/test split for '{tic}' — skipping.")
                continue

            model = LinearRegression()
            model.fit(x_train, y_train)
            pred = model.predict(x_test)

            res = pd.concat([
                pd.Series(pred, index=y_test.index, name='predicted'),
                y_test.rename('actual'),
            ], axis=1)
            res = res.merge(combined, left_index=True, right_index=True, how='outer').dropna()

            if res.empty:
                print(f"[Regression] No overlapping rows to score for '{tic}' — skipping.")
                continue

            r2 = r2_score(res['actual'], res['predicted'])
            print(f"[Regression] {tic}: R^2 = {r2:.4f}")

            fig = go.Figure()
            fig.add_trace(go.Scatter(x=res.index, y=res['actual'], mode='lines+markers', name=f'Actual {tic}'))
            fig.add_trace(go.Scatter(x=res.index, y=res['predicted'], mode='lines+markers', name='Predicted'))
            fig.show()

            res.to_csv(os.path.join(self.base_path, f'{tic}.csv'), header=True, index=True, mode='w+')



In [3]:
newspipe=NewsPipeline()

In [4]:
df,market_df,ticker_df=newspipe.run_acquire_stats()

                                                   title  \
0      Jet Fuel And Commercial LPG Prices Slashed, Of...   
1      OMCs bring new year cheer, announce rate cut f...   
2      Jet fuel price cut by 1.5%, commercial LPG rat...   
3      ATF price cut by 1.5%, commercial LPG rates do...   
4      5 EV Charging Infrastructure Stocks to Add to ...   
...                                                  ...   
75644  Clicking Suspicious Links Despite Security War...   
75645  From SBI, BEL to TVS Motor: Motilal Oswal’s to...   
75646  PhonePe SBI Card SELECT Black and PURPLE Deval...   
75647  RBI Repo Rate Should Hold at 5.25%, Says SBI C...   
75648  Customer Clicked A Suspicious Link: Can The Ba...   

                                             description    published date  \
0      Jet Fuel And Commercial LPG Prices Slashed, Of...  01-01-2025 08:00   
1      OMCs bring new year cheer, announce rate cut f...  01-01-2025 08:00   
2      Jet fuel price cut by 1.5%, commercial

In [5]:
ticker_df.tail(5)

,date,ticker,mean_sentiment,news_count,sentiment_volatility,mean_sentiment_logit,sentiment_volatility_logit,pert_bull,pert_neutral,pert_bearish,sentiment_volatility_filled,Zscore,Zscore_logit,sentiment_change,sentiment_change_logit
2034,31-10-2025,NVIDIA.csv,0.229723,88,0.348004,3.128800,2.060830,17.045455,80.681818,2.272727,0.348004,0.586429,0.901260,0.219427,2.054315
2035,31-10-2025,SBI.csv,-0.009665,26,0.431731,0.564745,2.634951,15.384615,69.230769,15.384615,0.431731,-0.081784,-0.268207,0.313011,1.669557
2036,31-12-2025,Bharat Petroleum Corporation.csv,0.315161,12,0.435485,3.384032,2.015396,33.333333,66.666667,0.000000,0.435485,0.664816,1.048218,-0.201309,-0.429395
2037,31-12-2025,NVIDIA.csv,0.157559,53,0.658879,1.438888,4.041045,43.396226,33.962264,22.641509,0.658879,0.200212,0.041433,-0.072164,-1.689913
2038,31-12-2025,SBI.csv,0.088816,26,0.441518,1.978725,2.710435,15.384615,76.923077,7.692308,0.441518,0.143081,0.260943,0.098482,1.413980


In [6]:
market_df.tail(5)

,mean_sentiment,news_count,sentiment_volatility,mean_sentiment_logit,sentiment_volatility_logit,pert_bull,pert_neutral,pert_bearish,sentiment_volatility_filled,Zscore,Zscore_logit,sentiment_change,sentiment_change_logit
date,,,,,,,,,,,,,
31-05-2026,-0.056792,126,0.447541,1.119998,3.036936,9.523810,73.809524,16.666667,0.447541,-0.144598,-0.010658,-0.221032,-0.789617
31-07-2025,-0.148378,116,0.467961,0.144280,3.206579,6.896552,64.655172,28.448276,0.467961,-0.334001,-0.314380,-0.091586,-0.975718
31-08-2025,-0.122073,66,0.576130,0.257501,3.643316,15.151515,48.484848,36.363636,0.576130,-0.225633,-0.245618,0.026305,0.113221
31-10-2025,0.246239,144,0.427052,2.808476,2.566285,24.305556,71.527778,4.166667,0.427052,0.558053,0.645334,0.368312,2.550976
31-12-2025,0.158701,91,0.576914,1.849630,3.519993,34.065934,50.549451,15.384615,0.576914,0.261355,0.198087,-0.087538,-0.958847


In [7]:
r=Regression2()
r.run_regression(t_df=ticker_df,ticker="Bharat Petroleum Corporation")

In [8]:
bpcl = ticker_df[
    ticker_df['ticker'].isin([
        'Bharat Petroleum Corporation',
        'Bharat Petroleum Corporation.csv',
        'BPCL.NS',
        'BPCL.NS.csv'
    ])
].copy()

bpcl['date'] = pd.to_datetime(
    bpcl['date'],
    format='mixed',
    dayfirst=True
)

bpcl = bpcl.sort_values('date')

In [9]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=bpcl['date'],
        y=bpcl['mean_sentiment'],
        mode='lines+markers',
        name='Mean Sentiment'
    )
)

fig.add_hline(y=0)

fig.update_layout(
    title='BPCL - Mean Sentiment Over Time',
    xaxis_title='Date',
    yaxis_title='Mean Sentiment',
    template='plotly_white'
)

fig.show()

In [10]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=bpcl['date'],
        y=bpcl['news_count'],
        mode='lines+markers',
        name='News Count'
    )
)

fig.update_layout(
    title='BPCL - News Volume Over Time',
    xaxis_title='Date',
    yaxis_title='News Count',
    template='plotly_white'
)

fig.show()

In [11]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=bpcl['date'],
        y=bpcl['sentiment_change'],
        mode='lines+markers',
        name='Sentiment Change'
    )
)

fig.add_hline(y=0)

fig.update_layout(
    title='BPCL - Sentiment Change Over Time',
    xaxis_title='Date',
    yaxis_title='Sentiment Change',
    template='plotly_white'
)

fig.show()

In [12]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=bpcl['news_count'],
        y=bpcl['mean_sentiment'],
        mode='markers',
        text=bpcl['date'].dt.strftime('%Y-%m-%d'),
        name='BPCL'
    )
)

fig.add_hline(y=0)

fig.update_layout(
    title='BPCL - News Volume vs Mean Sentiment',
    xaxis_title='News Count',
    yaxis_title='Mean Sentiment',
    template='plotly_white'
)

fig.show()